In [1]:
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import operator
import time

# 모델 일관성을 위한 상수 정의
MODEL_NAME = "qwen2.5:1.5b"
BASE_URL = "http://localhost:11434"
EMBEDDING_MODEL_NAME = "bge-m3:latest"

c:\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("=" * 60)
print(" 로컬 RAG 시스템 초기화")
print("=" * 60)

# 1. 문서 로딩
print("==> 1. 문서 로딩 → PDF 읽기...")
try:
    loader = PyPDFLoader('../data/tutorial-korean.pdf')
    documents = loader.load()
    print(f"총 {len(documents)}페이지 로드 완료")
except Exception as e:
    print(f"PDF 로딩 실패: {e}")
    print("'../data/tutorial-korean.pdf' 파일이 존재하는지 확인해주세요.")
    exit(1)

 로컬 RAG 시스템 초기화
==> 1. 문서 로딩 → PDF 읽기...
총 39페이지 로드 완료


In [3]:

# 2. 문서 분할
print("==> 2. 문서 분할 → 작은 청크로 나누기")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"{len(chunks)}개 청크 생성 완료")
print(f"평균 청크 길이: {sum(len(chunk.page_content) for chunk in chunks) / len(chunks):.0f}자")

# 3. 임베딩 설정
print("==> 3. 임베딩 설정...")
print("임베딩 모델 다운로드 중... (최초 실행 시 시간이 소요됩니다)")
try:
    embeddings = OllamaEmbeddings(
        model=EMBEDDING_MODEL_NAME,  # 사용할 임베딩 모델 이름
        base_url="http://localhost:11434",  # Ollama 서버 URL
    )
    print("OllamaEmbeddings 임베딩 모델 설정 완료")
except Exception as e:
    print(f"임베딩 모델 설정 실패: {e}")
    print("인터넷 연결을 확인하거나 다음 패키지를 설치해주세요:")
    #print("pip install sentence-transformers")
    exit(1)

==> 2. 문서 분할 → 작은 청크로 나누기
77개 청크 생성 완료
평균 청크 길이: 594자
==> 3. 임베딩 설정...
임베딩 모델 다운로드 중... (최초 실행 시 시간이 소요됩니다)
OllamaEmbeddings 임베딩 모델 설정 완료


C:\Users\vega2\AppData\Local\Temp\ipykernel_26244\727932669.py:17: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


In [4]:

# 4. 벡터스토어 생성
print("==> 4. 벡터스토어 생성...")
print("임베딩 생성 중... (시간이 소요됩니다)")

start_time = time.time()
try:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    embedding_time = time.time() - start_time
    print(f"FAISS 벡터스토어 생성 완료 ({len(chunks)}개 벡터)")
    print(f"임베딩 소요 시간: {embedding_time:.1f}초")
except Exception as e:
    print(f"벡터스토어 생성 실패: {e}")
    print("모델이 설치되어 있는지 확인해주세요: ollama pull qwen2.5:latest")
    exit(1)

==> 4. 벡터스토어 생성...
임베딩 생성 중... (시간이 소요됩니다)
FAISS 벡터스토어 생성 완료 (77개 벡터)
임베딩 소요 시간: 259.0초


In [5]:

# 5. 검색기 설정
print("==> 5. 검색기 설정...")
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
print("검색기 설정 완료")

# 6. LLM 설정
print("==> 6. LLM 설정...")
try:
    llm = ChatOllama(
        model=MODEL_NAME,
        temperature=0.1,
        num_predict=800
    )
    print("LLM 설정 완료")
except Exception as e:
    print(f"LLM 설정 실패: {e}")
    exit(1)


==> 5. 검색기 설정...
검색기 설정 완료
==> 6. LLM 설정...
LLM 설정 완료


In [7]:

# 7. 프롬프트 설정 (ChatPromptTemplate 사용)
print("==> 7. 프롬프트 설정...")
prompt_template = """당신은 BlueJ 프로그래밍 환경 전문가입니다.
아래 문서 내용을 바탕으로 정확하고 친절한 답변을 제공해주세요.

문서 내용:
{context}

질문: {question}

답변 규칙:
1. 문서 내용만을 근거로 답변하세요
2. 단계별로 설명하세요  
3. 구체적인 메뉴명, 버튼명을 포함하세요
4. 문서에 없는 정보는 "문서에서 찾을 수 없습니다"라고 하세요

답변:"""

prompt = ChatPromptTemplate.from_template(prompt_template)
print("프롬프트 설정 완료")


==> 7. 프롬프트 설정...
프롬프트 설정 완료


In [8]:

# 8. LCEL RAG 파이프라인 생성 (RunnablePassthrough 사용)
print("==> 8. LCEL RAG 파이프라인 생성...")
try:
    # 문서 검색 → 프롬프트 형식화 → LLM 호출 → 파싱
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    
    # 소스 문서 반환을 위한 별도 체인
    def get_sources(inputs):
        docs = retriever.invoke(inputs["question"])
        return {
            "answer": rag_chain.invoke(inputs["question"]),
            "source_documents": docs
        }
    
    qa_chain = lambda inputs: get_sources({"question": inputs["query"]})
    
    print("LCEL RAG 파이프라인 구축 완료 (RunnablePassthrough 사용)")
except Exception as e:
    print(f"LCEL 체인 생성 실패: {e}")
    exit(1)

==> 8. LCEL RAG 파이프라인 생성...
LCEL RAG 파이프라인 구축 완료 (RunnablePassthrough 사용)


In [9]:

# 9. 테스트 실행 (기존 출력 형식 동일)
print("\n" + "=" * 60)
print(" 로컬 RAG 시스템 테스트")
print("=" * 60)

test_questions = [
    "BlueJ에서 객체를 생성하는 방법은 무엇인가요?",
    "컴파일 오류가 발생했을 때 어떻게 확인할 수 있나요?"
]

total_start_time = time.time()

for i, question in enumerate(test_questions, 1):
    print(f"\n【테스트 {i}/{len(test_questions)}】")
    print(f"질문: {question}")
    print("답변 생성 중...")
    
    question_start_time = time.time()
    
    try:
        result = qa_chain({"query": question})
        answer = result["answer"]
        source_docs = result["source_documents"]
        
        question_time = time.time() - question_start_time
        
        print(f"\n답변: (응답시간: {question_time:.1f}초)")
        print("-" * 50)
        print(answer)
        
        print(f"\n참조 문서:")
        for j, doc in enumerate(source_docs[:2], 1):
            page = doc.metadata.get('page', 'N/A')
            preview = doc.page_content[:50].replace('\n', ' ')
            print(f"   {j}. 페이지 {page}: {preview}...")
            
    except Exception as e:
        print(f"답변 생성 실패: {e}")
    
    print("\n" + "-" * 40)

total_time = time.time() - total_start_time
print(f"\n테스트 완료! (총 소요시간: {total_time:.1f}초)")



 로컬 RAG 시스템 테스트

【테스트 1/2】
질문: BlueJ에서 객체를 생성하는 방법은 무엇인가요?
답변 생성 중...

답변: (응답시간: 56.6초)
--------------------------------------------------
BlueJ에서 객체를 생성하는 방법은 다음과 같습니다:

1. **클래스 팝업 메뉴 선택**: 
   - 클릭한 클래스의 이름이 표시된 클래스 팝업 메뉴를 선택합니다.
   - 예: Person 클래스의 경우, "Person"을 클릭하면 Person 클래스의 모든 메소드와 생성자 함수가 나타납니다.

2. **클래스 생성기 사용**:
   - Tools > Use Libraries Class 메뉴를 선택합니다.
   - 팝업 대화상자를 열면, Java 표준 라이브러리에서 제공하는 클래스들을 선택할 수 있습니다.
     - 예: String 클래스의 객체를 생성하려면 "java.lang.String"을 입력하고 엔터를 누릅니다.

3. **프로젝트 열기**:
   - Project > Open... 메뉴를 선택합니다.
   - 프로젝트 디렉토리를 선택하여 프로젝트를 열어보세요.
     - 예: people 프로젝트를 열면, 프로젝트의 모든 파일들이 표시됩니다.

4. **사용자 정의 클래스 생성**:
   - 직접 사용자 정의 클래스를 작성하고, Tools > Use Libraries Class 메뉴에서 해당 클래스를 선택하여 객체를 생성할 수 있습니다.
     - 예: "MyClass"라는 이름으로 클래스를 작성한 후, "Tools > Use Libraries Class" 메뉴를 통해 생성합니다.

5. **사용자 정의 메소드 호출**:
   - 직접 사용자 정의 메소드를 호출하려면, 해당 메소드가 포함된 클래스 팝업 메뉴를 선택하고, 메소드 이름을 클릭하여 실행할 수 있습니다.
     - 예: "MyClass.myMethod()"를 호출하려면, "MyClass" 클래스 팝업 메뉴를 선택하고 "myMeth

In [ ]:

# 10. 대화형 모드 (기존 형식 동일)
print("\n" + "=" * 60)
print(" 대화형 모드 (종료: 'quit' 입력)")
print("=" * 60)

while True:
    try:
        user_question = input("\n질문을 입력하세요: ").strip()
        
        if user_question.lower() in ['quit', 'exit', '종료', 'q']:
            print("RAG 시스템을 종료합니다.")
            break
            
        if not user_question:
            continue
            
        print("답변 생성 중...")
        start_time = time.time()
        
        result = qa_chain({"query": user_question})
        answer = result["answer"]
        response_time = time.time() - start_time
        
        print(f"\n답변: (응답시간: {response_time:.1f}초)")
        print("-" * 50)
        print(answer)
        
    except KeyboardInterrupt:
        print("\nRAG 시스템을 종료합니다.")
        break
    except Exception as e:
        print(f"오류 발생: {e}")

print("\n로컬 RAG 시스템 세션 종료!")


 대화형 모드 (종료: 'quit' 입력)
답변 생성 중...

답변: (응답시간: 54.4초)
--------------------------------------------------
애플릿은 BlueJ 프로그래밍 환경에서 사용되는 액세스 토근입니다. 애플릿은 일반적인 클래스와 비슷하게 만들어진 새로운 클래스를 포함하고 있으며, 이는 애플릿이 실행될 수 있도록 기본적인 코드가 포함되어 있습니다.

애플릿의 메소드들은 각각 그 목적을 설명해놓은 주석을 가지고 있으며, 샘플 코드는 모두 paint 메소드 안에 있습니다. 애플릿은 일반적으로 2줄의 텍스트로 이루어진 간단한 애플릿을 보여줍니다.

애플릿이 실행될 때, 애플릿 객체를 다른 클래스처럼 생성하는 것은 매우 유용합니다. 애플릿의 팝업 메뉴에서 볼 수 있습니다. 이는 애플릿의 구현 부분에 작성한 한 개의 메소드를 테스트하기 위해 사용됩니다.

애플릿은 일반적으로 중단점을 설정했을 때, 애플릿 뷰어나 웹 브라우저에서 애플릿을 실행할 수 있습니다. 그러나 이러한 애플릿의 테스트는 오브젝트 벤치에서 완벽하게 이루어지지 않을 수 있습니다.

애플릿은 일반적으로 메뉴, 창, 팝업 메뉴 등의 요소를 포함하며, 이들은 사용자가 애플릿을 조작할 때 필요한 기능을 제공합니다. 예를 들어, 메뉴는 애플릿의 기본 설정을 변경하거나 특정 작업을 수행하기 위한 방법을 제공합니다.

애플릿은 일반적으로 프로그래밍 환경에서 중요한 역할을 합니다. 이는 사용자가 애플릿을 실행하고 그에 대한 응답을 기다리거나, 애플릿이 일정 시간 동안 유지될 수 있도록 설정하는 등의 작업을 수행합니다.
답변 생성 중...

답변: (응답시간: 48.1초)
--------------------------------------------------
애플릿은 BlueJ 프로그래밍 환경에서 사용되는 기본적인 액션입니다. 애플릿은 특정 작업이나 기능을 수행하는 클래스를 포함한 새로운 클래스로 만듭니다.

1. **새 클래스 만들기**:
   - `New Cl